
Building a YouTube assistant with LangChain that searches videos, retrieves metadata and transcripts, and generates summaries. Implementing manual tool calling, fixed-sequence chains, and recursive workflows to support dynamic tool selection and multi-step queries.

### Installing required libraries


In [1]:
%%capture
%pip install pytube 
%pip install youtube-transcript-api==1.1.0
%pip install langchain-community==0.3.16
%pip install langchain==0.3.23
%pip install langchain-openai==0.3.14
%pip install yt-dlp

### Importing required libraries


In [2]:
import re
from pytube import YouTube
from langchain_core.tools import tool
from IPython.display import display, JSON
import yt_dlp
from typing import List, Dict
from langchain_core.messages import HumanMessage
from langchain_core.messages import ToolMessage
import json

# Suppress warnings
import warnings
warnings.filterwarnings("ignore")

# Suppress pytube errors
import logging
pytube_logger = logging.getLogger('pytube')
pytube_logger.setLevel(logging.ERROR)

# Suppress yt-dlp warnings
yt_dpl_logger = logging.getLogger('yt_dlp')
yt_dpl_logger.setLevel(logging.ERROR)

Initializing GPT-4o-mini through LangChain’s OpenAI integration to interpret user queries and select the appropriate tools for YouTube search, content retrieval, and summarization.

In [3]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("gpt-4o-mini", model_provider="openai")

# API Disclaimer

In [ ]:

os.environ["OPENAI_API_KEY"] = "OpenAI API key here"

# Tools


In [4]:
@tool
def extract_video_id(url: str) -> str:
    """
    Extracts the 11-character YouTube video ID from a URL.
    
    Args:
        url (str): A YouTube URL containing a video ID.

    Returns:
        str: Extracted video ID or error message if parsing fails.
    """
    
    # Regex pattern to match video IDs
    pattern = r'(?:v=|be/|embed/)([a-zA-Z0-9_-]{11})'
    match = re.search(pattern, url)
    return match.group(1) if match else "Error: Invalid YouTube URL"

In [5]:
print(extract_video_id.name)
print("----------------------------")
print(extract_video_id.description)
print("----------------------------")
print(extract_video_id.func)

extract_video_id
----------------------------
Extracts the 11-character YouTube video ID from a URL.

Args:
    url (str): A YouTube URL containing a video ID.

Returns:
    str: Extracted video ID or error message if parsing fails.
----------------------------
<function extract_video_id at 0x7a3789792f20>

In [6]:
extract_video_id.run("https://www.youtube.com/watch?v=hfIUstzHs9A")

'hfIUstzHs9A'

In [7]:
extract_video_id

StructuredTool(name='extract_video_id', description='Extracts the 11-character YouTube video ID from a URL.\n\nArgs:\n    url (str): A YouTube URL containing a video ID.\n\nReturns:\n    str: Extracted video ID or error message if parsing fails.', args_schema=<class 'langchain_core.utils.pydantic.extract_video_id'>, func=<function extract_video_id at 0x7a3789792f20>)

## Tool list 
Organizing custom tools in a tools list, starting with extract_video_id, and making them available to the model through llm.bind_tools(tools). The list collects available tools; the model determines which to call based on the query.



In [8]:
tools = []
tools.append(extract_video_id)

### Defining transcript fetching tool

Creating a transcript retrieval tool with YouTubeTranscriptApi that accepts a video ID and an optional language parameter. Combining caption segments into a single text string and returning a descriptive error if retrieval fails.


In [11]:
from youtube_transcript_api import YouTubeTranscriptApi


@tool
def fetch_transcript(video_id: str, language: str = "en") -> str:
    """
    Fetches the transcript of a YouTube video.
    
    Args:
        video_id (str): The YouTube video ID (e.g., "dQw4w9WgXcQ").
        language (str): Language code for the transcript (e.g., "en", "es").
    
    Returns:
        str: The transcript text or an error message.
    """
    
    try:
        ytt_api = YouTubeTranscriptApi()
        transcript = ytt_api.fetch(video_id, languages=[language])
        return " ".join([snippet.text for snippet in transcript.snippets])
    except Exception as e:
        return f"Error: {str(e)}"

In [12]:
fetch_transcript.run("hfIUstzHs9A")

'Over the past couple of months, large language models, or LLMs, such as chatGPT, have taken the world by storm. Whether it\'s writing poetry or helping plan your upcoming vacation, we are seeing a step change in the performance of AI and its potential to drive enterprise value. My name is Kate Soule. I\'m a senior manager of business strategy at IBM Research, and today I\'m going to give a brief overview of this new field of AI that\'s emerging and how it can be used in a business setting to drive value. Now, large language models are actually a part of a different class of models called foundation models. Now, the term "foundation models" was actually first coined by a team from Stanford when they saw that the field of AI was converging to a new paradigm. Where before AI applications were being built by training, maybe a library of different AI models, where each AI model was trained on very task-specific data to perform very specific task. They predicted that we were going to start 

---
Adding the `fetch_transcript` tool to your tools list.


In [13]:
tools.append(fetch_transcript)

### Defining YouTube search tool

Creating a YouTube search tool with PyTube’s Search class that accepts a query and returns matching videos with their titles, video IDs, and shortened URLs.


In [14]:
from pytube import Search
from langchain.tools import tool
from typing import List, Dict

@tool
def search_youtube(query: str) -> List[Dict[str, str]]:
    """
    Search YouTube for videos matching the query.
    
    Args:
        query (str): The search term to look for on YouTube
        
    Returns:
        List of dictionaries containing video titles and IDs in format:
        [{'title': 'Video Title', 'video_id': 'abc123'}, ...]
        Returns error message if search fails
    """
    try:
        s = Search(query)
        return [
            {
                "title": yt.title,
                "video_id": yt.video_id,
                "url": f"https://youtu.be/{yt.video_id}"
            }
            for yt in s.results
        ]
    except Exception as e:
        return f"Error: {str(e)}"

In [15]:
search_out=search_youtube.run("Generative AI")
display(JSON(search_out))

<IPython.core.display.JSON object>

Appending the `search_youtube` tool to tools list.


In [16]:
tools.append(search_youtube)

### Defining metadata extraction tool

Creating a metadata extraction tool with yt-dlp that accepts a YouTube URL and returns available video details, including title, duration, channel, engagement counts, and chapter markers.


In [17]:
@tool
def get_full_metadata(url: str) -> dict:
    """Extract metadata given a YouTube URL, including title, views, duration, channel, likes, comments, and chapters."""
    with yt_dlp.YoutubeDL({'quiet': True, 'logger': yt_dpl_logger}) as ydl:
        info = ydl.extract_info(url, download=False)
        return {
            'title': info.get('title'),
            'views': info.get('view_count'),
            'duration': info.get('duration'),
            'channel': info.get('uploader'),
            'likes': info.get('like_count'),
            'comments': info.get('comment_count'),
            'chapters': info.get('chapters', [])
        }

In [18]:
meta_data=get_full_metadata.run("https://www.youtube.com/watch?v=T-D1OfcDW1M")
display(JSON(meta_data))

<IPython.core.display.JSON object>

Adding the `get_full_metadata` tool to your tools list.


In [19]:
tools.append(get_full_metadata)

### Defining thumbnail retrieval tool

Creating a thumbnail retrieval tool with yt-dlp that returns available thumbnail URLs, dimensions, and formatted resolutions for a YouTube video.


In [20]:
@tool
def get_thumbnails(url: str) -> List[Dict]:
    """
    Get available thumbnails for a YouTube video using its URL.
    
    Args:
        url (str): YouTube video URL (any format)
        
    Returns:
        List of dictionaries with thumbnail URLs and resolutions in YouTube's native order
    """
    
    try:
        with yt_dlp.YoutubeDL({'quiet': True, 'logger': yt_dpl_logger}) as ydl:
            info = ydl.extract_info(url, download=False)
            
            thumbnails = []
            for t in info.get('thumbnails', []):
                if 'url' in t:
                    thumbnails.append({
                        "url": t['url'],
                        "width": t.get('width'),
                        "height": t.get('height'),
                        "resolution": f"{t.get('width', '')}x{t.get('height', '')}".strip('x')
                    })
            
            return thumbnails

    except Exception as e:
        return [{"error": f"Failed to get thumbnails: {str(e)}"}]

In [21]:
thumbnails=get_thumbnails.run("https://www.youtube.com/watch?v=qWHaMrR5WHQ")

display(JSON(thumbnails))

<IPython.core.display.JSON object>

In [22]:
tools.append(get_thumbnails)

##  Binding tools


Binding the YouTube tools to the language model with llm.bind_tools(tools), enabling it to select tools and generate appropriate arguments based on user requests.

In [23]:
llm_with_tools = llm.bind_tools(tools)

In [24]:
for tool in tools:
    schema = {
   "name": tool.name,
   "description": tool.description,
   "parameters": tool.args_schema.schema() if tool.args_schema else {},
   "return": tool.return_type if hasattr(tool, "return_type") else None}
    display(JSON(schema))
    

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

In [25]:
query = "I want to summarize youtube video: https://www.youtube.com/watch?v=T-D1OfcDW1M in english"
print(query)

I want to summarize youtube video: https://www.youtube.com/watch?v=T-D1OfcDW1M in english

Wrapping the user query in a HumanMessage object to include it in the conversation passed to the language model.

In [26]:
messages = [HumanMessage(content = query)]
print(messages)

[HumanMessage(content='I want to summarize youtube video: https://www.youtube.com/watch?v=T-D1OfcDW1M in english', additional_kwargs={}, response_metadata={})]

### LangChain tool binding process

Invoking the tool-bound model with a YouTube summarization query and storing its reply in response_1, including any requested tool calls and their input arguments.


In [27]:
response_1 = llm_with_tools.invoke(messages)
response_1

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_LA0guGtHUVjU1VXvlukltyYP', 'function': {'arguments': '{"url":"https://www.youtube.com/watch?v=T-D1OfcDW1M"}', 'name': 'extract_video_id'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 380, 'total_tokens': 409, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_0d55417811', 'id': 'chatcmpl-EbjuNxypMLyynGyT6yLWe5TERszV5', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--01a1630f-60c1-7228-b27d-8c6b3776fe03-0', tool_calls=[{'name': 'extract_video_id', 'args': {'url': 'https://www.youtube.com/watch?v=T-D1OfcDW1M'}, 'id': 'call_LA0guGtHUVjU1VXvlukltyYP', 'type': 'tool_call'}], usage_metadata={'input_tokens': 380, 'output_toke

In [28]:
messages.append(response_1)

In [30]:
tool_mapping = {
    "get_thumbnails" : get_thumbnails,
    "extract_video_id": extract_video_id,
    "fetch_transcript": fetch_transcript,
    "search_youtube": search_youtube,
    "get_full_metadata": get_full_metadata
}

In [31]:
tool_calls_1 = response_1.tool_calls
display(JSON(tool_calls_1))

<IPython.core.display.JSON object>

In [32]:
tool_name=tool_calls_1[0]['name']
print(tool_name)

extract_video_id

In [33]:
tool_call_id =tool_calls_1[0]['id']
print(tool_call_id)

call_LA0guGtHUVjU1VXvlukltyYP

In [34]:
args=tool_calls_1[0]['args']
print(args)

{'url': 'https://www.youtube.com/watch?v=T-D1OfcDW1M'}

In [35]:
my_tool=tool_mapping[tool_calls_1[0]['name']]

In [36]:
video_id =my_tool.invoke(tool_calls_1[0]['args'])
video_id

'T-D1OfcDW1M'

In [37]:
messages.append(ToolMessage(content = video_id, tool_call_id = tool_calls_1[0]['id']))

Passing the updated conversation, including the extracted video ID, back to the model so it can select the next tool needed to summarize the video.

In [38]:
response_2 = llm_with_tools.invoke(messages)
response_2

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_lJ6sLQNkzfY74tuE5ip5jhVB', 'function': {'arguments': '{"video_id":"T-D1OfcDW1M","language":"en"}', 'name': 'fetch_transcript'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 426, 'total_tokens': 453, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_0d55417811', 'id': 'chatcmpl-EbjvPqsfuAzqLVnotftiEL2cmbYLB', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--01a16310-5c20-706b-8cbd-b016c133f536-0', tool_calls=[{'name': 'fetch_transcript', 'args': {'video_id': 'T-D1OfcDW1M', 'language': 'en'}, 'id': 'call_lJ6sLQNkzfY74tuE5ip5jhVB', 'type': 'tool_call'}], usage_metadata={'input_tokens': 426, 'output_tokens': 27, 'total_toke

In [39]:
messages.append(response_2)

In [40]:
tool_calls_2 = response_2.tool_calls
tool_calls_2

[{'name': 'fetch_transcript',
  'args': {'video_id': 'T-D1OfcDW1M', 'language': 'en'},
  'id': 'call_lJ6sLQNkzfY74tuE5ip5jhVB',
  'type': 'tool_call'}]

In [41]:
fetch_transcript_tool_output = tool_mapping[tool_calls_2[0]['name']].invoke(tool_calls_2[0]['args'])
fetch_transcript_tool_output

'Large language models. They are everywhere. They get some things amazingly right and other things very interestingly wrong. My name\xa0is Marina Danilevsky. I am a Senior Research Scientist here at IBM Research. And I want\xa0to tell you about a framework to help large language models be more accurate and more up to\xa0date: Retrieval-Augmented Generation, or RAG. Let\'s just talk about the "Generation" part for a\xa0minute. So forget the "Retrieval-Augmented". So the\xa0generation, this refers to large language models,\xa0or LLMs, that generate text in response to a user query, referred to as a prompt. These\xa0models can have some undesirable behavior. I want to tell you an anecdote to illustrate this. So my kids, they recently asked me this question: "In our solar system, what planet has the most\xa0moons?" And my response was, “Oh, that\'s really great that you\'re asking this question. I loved\xa0space when I was your age.” Of course, that was like 30 years ago. But I know this! 

---
You're adding the transcript content to your conversation history by creating another `ToolMessage` that contains the transcript text and the ID of the tool call that requested it. This gives the LLM access to the actual video content so it can generate a summary.


In [42]:
messages.append(ToolMessage(content = fetch_transcript_tool_output, tool_call_id = tool_calls_2[0]['id']))

Passing the complete conversation history, including the video ID and retrieved transcript, to the model to generate the requested video summary.

In [44]:
summary = llm_with_tools.invoke(messages)

In [45]:
summary

AIMessage(content="The video features Marina Danilevsky, a Senior Research Scientist at IBM Research, discussing the challenges and solutions related to large language models (LLMs), particularly through a framework called Retrieval-Augmented Generation (RAG).\n\n### Summary:\n1. **Introduction to LLMs**: Danilevsky explains that while LLMs can produce accurate responses, they often struggle with two main challenges: not sourcing their information and being out of date.\n\n2. **Anecdote**: She shares a personal story about her children asking which planet has the most moons. She initially states Jupiter, citing a past article without current validation, illustrating how LLMs can confidently provide incorrect answers without reliable sources.\n\n3. **RAG Framework**: RAG improves LLMs by incorporating a retrieval step. Instead of solely relying on pre-existing knowledge, LLMs first query an external content store (e.g., the internet or specific documents) to obtain the most relevant and

Automating tool execution by looking up each requested tool in tool_mapping, invoking it with the supplied arguments, and returning its output as a ToolMessage. Including tool_call_id links each result to the corresponding model request.

In [46]:
# Define the processing steps
def execute_tool(tool_call):
    """Execute single tool call and return ToolMessage"""
    try:
        result = tool_mapping[tool_call["name"]].invoke(tool_call["args"])
        return ToolMessage(
            content=str(result),
            tool_call_id=tool_call["id"]
        )
    except Exception as e:
        return ToolMessage(
            content=f"Error: {str(e)}",
            tool_call_id=tool_call["id"]
        )

        

In [47]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda


## Building the summarization chain

Building summarization_chain with the | operator to sequence message formatting, model calls, tool execution, and conversation updates. Using RunnablePassthrough to preserve state and RunnableLambda to extract the final summary text.


In [48]:
summarization_chain = (
    # Start with initial query
    RunnablePassthrough.assign(
        messages=lambda x: [HumanMessage(content=x["query"])]
    )
    # First LLM call (extract video ID)
    | RunnablePassthrough.assign(
        ai_response=lambda x: llm_with_tools.invoke(x["messages"])
    )
    # Process first tool call
    | RunnablePassthrough.assign(
        tool_messages=lambda x: [
            execute_tool(tc) for tc in x["ai_response"].tool_calls
        ]
    )
    # Update message history
    | RunnablePassthrough.assign(
        messages=lambda x: x["messages"] + [x["ai_response"]] + x["tool_messages"]
    )
    # Second LLM call (fetch transcript)
    | RunnablePassthrough.assign(
        ai_response2=lambda x: llm_with_tools.invoke(x["messages"])
    )
    # Process second tool call
    | RunnablePassthrough.assign(
        tool_messages2=lambda x: [
            execute_tool(tc) for tc in x["ai_response2"].tool_calls
        ]
    )
    # Final message update
    | RunnablePassthrough.assign(
        messages=lambda x: x["messages"] + [x["ai_response2"]] + x["tool_messages2"]
    )
    # Generate final summary
    | RunnablePassthrough.assign(
        summary=lambda x: llm_with_tools.invoke(x["messages"]).content
    )
    # Return just the summary text
    | RunnableLambda(lambda x: x["summary"])
)


In [49]:
# Usage
result = summarization_chain.invoke({
    "query": "Summarize this YouTube video: https://www.youtube.com/watch?v=1bUy-1hGZpI"
})

print("Video Summary:\n", result)

Video Summary:
 The video discusses LangChain, an open-source framework designed for developing applications that utilize large language models (LLMs). LangChain allows for the integration of multiple LLMs in business applications, catering to different tasks such as interpreting user queries and generating responses.

### Key Points:
- **Components of LangChain**:
  - **LLM Module**: Provides a standard interface for various LLMs, allowing users to choose models like GPT-4 or LLaMA 2.
  - **Prompts**: The framework offers a way to create prompt templates, allowing dynamic instruction to models without hardcoding.
  - **Chains**: These are sequences of functions that connect different components, enabling complex workflows.
  - **Document Loaders**: Tools to import data from various sources (e.g., Google Drive, YouTube transcripts).
  - **Memory Utilities**: Enhance LLMs by providing long-term memory for previous interactions.
  - **Agents**: Use LLMs as reasoning engines to take actio

Using RunnablePassthrough.assign to add a message list containing the query as a HumanMessage, while preserving the original input dictionary.

In [50]:
initial_setup = RunnablePassthrough.assign(
    messages=lambda x: [HumanMessage(content=x["query"])]
)

In [51]:
first_llm_call = RunnablePassthrough.assign(
    ai_response=lambda x: llm_with_tools.invoke(x["messages"])
)

In [52]:
first_tool_processing = RunnablePassthrough.assign(
    tool_messages=lambda x: [
        execute_tool(tc) for tc in x["ai_response"].tool_calls
    ]
).assign(
    messages=lambda x: x["messages"] + [x["ai_response"]] + x["tool_messages"]
)

In [53]:
second_llm_call = RunnablePassthrough.assign(
    ai_response2=lambda x: llm_with_tools.invoke(x["messages"])
)

In [54]:
second_tool_processing = RunnablePassthrough.assign(
    tool_messages2=lambda x: [
        execute_tool(tc) for tc in x["ai_response2"].tool_calls
    ]
).assign(
    messages=lambda x: x["messages"] + [x["ai_response2"]] + x["tool_messages2"]
)

#### Generating the final summary
Generating the video summary by passing the complete message history to the model, then using RunnableLambda to extract and return only the response content.


In [55]:
final_summary = RunnablePassthrough.assign(
    summary=lambda x: llm_with_tools.invoke(x["messages"]).content
) | RunnableLambda(lambda x: x["summary"])

In [56]:
chain = (
    initial_setup
    | first_llm_call
    | first_tool_processing
    | second_llm_call
    | second_tool_processing
    | final_summary
)

In [57]:
query = {"query": "I want to summarize youtube video: https://www.youtube.com/watch?v=T-D1OfcDW1M in english"}
result = summarization_chain.invoke(query)
print("Video Summary:\n", result)

Video Summary:
 The video features Marina Danilevsky, a Senior Research Scientist at IBM Research, discussing a framework called Retrieval-Augmented Generation (RAG) that aims to enhance the accuracy and timeliness of large language models (LLMs).

Key points from the video include:

1. **Challenges of LLMs**: LLMs often exhibit two main problems: they can provide answers without citing sources, and their information can become outdated. Danilevsky illustrates this with a personal anecdote about answering a question regarding moons in our solar system, where she mistakenly cites Jupiter instead of the correct answer, Saturn.

2. **Retrieval-Augmented Generation**: RAG addresses the shortcomings of LLMs by incorporating a retrieval component. Instead of solely relying on training data, LLMs can access a content store (which can be open like the internet or closed like specific documents). This allows the model to pull current and relevant information before generating an answer.

3. **W

#### Testing the Chain with a Different Query

Testing the chain with a query about trending videos by passing a query dictionary to .invoke(), checking whether its tool-calling workflow can handle a request beyond video summarization.


In [58]:
query = {"query": "Get top 3 youtube videos in Finland and their metadata"}
try:
    result = summarization_chain.invoke(query)
    print("Video Summary:\n", result)
except Exception as e:
    print("Non-critical network error:", e)

Video Summary:
 Here are the top 3 YouTube videos related to Finland along with their metadata:

1. **Video Title**: [Wonders of Finland | The Most Amazing Places in Finland | Travel Video 4K](https://youtu.be/TPdt07geBrA)
   - **Views**: 699,820
   - **Duration**: 49 minutes and 43 seconds
   - **Channel**: Trip Insight
   - **Likes**: 9,447
   - **Comments**: 445
   - **Chapters**: 
     - Welcome to Finland (0:00 - 4:28)
     - Suomenlinna Fortress (4:28 - 6:21)
     - Helsinki Cathedral (6:21 - 8:03)
     - ... (additional chapters available)

2. **Video Title**: [Amazing Places to visit in Finland - Travel Video](https://youtu.be/996NUuxOhOI)
   - **Views**: 183,612
   - **Duration**: 9 minutes and 9 seconds
   - **Channel**: Joyous Travel
   - **Likes**: 5,229
   - **Comments**: 93
   - **Chapters**: 
     - Is Finland Worth Visiting? (0:00 - 0:33)
     - Helsinki (0:33 - 1:22)
     - Savonlinna (1:22 - 2:25)
     - ... (additional chapters available)

3. **Video Title**: [Finlan

In [56]:
result

'Here are the top 3 YouTube videos related to Finland along with their metadata:\n\n1. **Amazing Places to visit in Finland - Travel Video**\n   - **Views:** 183,612\n   - **Duration:** 9 minutes and 9 seconds\n   - **Channel:** Joyous Travel\n   - **Likes:** 5,229\n   - **Comments:** 93\n   - **Chapters:**\n     - Is Finland Worth Visiting? (0:00 - 0:33)\n     - Helsinki (0:33 - 1:22)\n     - Savonlinna (1:22 - 2:25)\n     - Turku (2:25 - 3:01)\n     - Porvoo (3:01 - 3:45)\n     - Åland Archipelago (3:45 - 4:49)\n     - Tampere (4:49 - 5:25)\n     - Kemi (5:25 - 6:20)\n     - Rovaniemi (6:20 - 7:04)\n     - Levi (7:04 - 7:40)\n     - Lemmenjoki National Park (7:40 - 8:29)\n     - Reason You Need to Visit Finland (8:29 - 9:09)\n   - **Video Link:** [Watch Here](https://youtu.be/996NUuxOhOI)\n\n2. **Top 10 Best Tourist Attractions in Helsinki, Finland - Travel Video 2026**\n   - **Views:** 1,294\n   - **Duration:** 8 minutes and 36 seconds\n   - **Channel:** Travel Insights\n   - **Like

## Recursive chain flow


Building a recursive chain to support a variable number of tool calls, extending the fixed two-step workflow to handle queries such as searching for videos and retrieving transcripts for multiple results.


In [59]:
from langchain_core.runnables import RunnableBranch, RunnableLambda
from langchain_core.messages import HumanMessage, ToolMessage
import json

def execute_tool(tool_call):
    """Execute single tool call and return ToolMessage"""
    try:
        result = tool_mapping[tool_call["name"]].invoke(tool_call["args"])
        content = json.dumps(result) if isinstance(result, (dict, list)) else str(result)
    except Exception as e:
        content = f"Error: {str(e)}"
    
    return ToolMessage(
        content=content,
        tool_call_id=tool_call["id"]
    )

Defining the recursive chain’s processing function to execute the latest message’s tool calls in parallel with execute_tool, append their results to the conversation, and invoke the model again. Returning the updated history with the tool results and the model’s next response.


In [60]:
def process_tool_calls(messages):
    """Recursive tool call processor"""
    last_message = messages[-1]
    
    # Execute all tool calls in parallel
    tool_messages = [
        execute_tool(tc) 
        for tc in getattr(last_message, 'tool_calls', [])
    ]
    
    # Add tool responses to message history
    updated_messages = messages + tool_messages
    
    # Get next LLM response
    next_ai_response = llm_with_tools.invoke(updated_messages)
    
    return updated_messages + [next_ai_response]

Defining the stopping condition by using getattr() to check the latest message for tool calls, returning True to continue processing or False when no further tools are requested.

In [61]:
def should_continue(messages):
    """Check if you need another iteration"""
    last_message = messages[-1]
    return bool(getattr(last_message, 'tool_calls', None))

Implementing recursion with should_continue and process_tool_calls to execute tools and update the conversation until no further tool calls are requested. Wrapping the function in RunnableLambda to integrate it into the LangChain workflow.

In [62]:
def _recursive_chain(messages):
    """Recursively process tool calls until completion"""
    if should_continue(messages):
        new_messages = process_tool_calls(messages)
        return _recursive_chain(new_messages)
    return messages

recursive_chain = RunnableLambda(_recursive_chain)

Assembling universal_chain to format the query as a HumanMessage, append the tool-bound model’s initial response, and pass the conversation to the recursive chain. Supporting a variable sequence of tool calls until the model returns a final response.


In [63]:
universal_chain = (
    RunnableLambda(lambda x: [HumanMessage(content=x["query"])])
    | RunnableLambda(lambda messages: messages + [llm_with_tools.invoke(messages)])
    | recursive_chain
)

In [64]:
query_us = {"query": "Show top 3 Spain trending videos with metadata and thumbnails"}

try:
    response = universal_chain.invoke(query_us)
    print("\nEurope Trending Videos:\n", response[-1])
except Exception as e:
    print("Non-critical network error while fetching Europe trending videos:", e)


Europe Trending Videos:
 content="Here are the top 3 trending videos in Spain along with their metadata and thumbnails:\n\n### 1. **10 Best Places to Visit in Spain - Travel Video**\n- **Channel:** touropia\n- **Views:** 3,205,178\n- **Duration:** 16 minutes and 40 seconds\n- **Likes:** 57,319\n- **Comments:** 1,300\n- **Chapters:**\n  - Intro (0:00 - 1:05)\n  - Santiago de Compostela (1:05 - 2:32)\n  - Toledo (2:32 - 4:06)\n  - Cordoba (4:06 - 5:43)\n  - San Sebastian (5:43 - 6:58)\n  - Valencia (6:58 - 8:35)\n  - Seville (8:35 - 10:13)\n  - Madrid (10:13 - 11:50)\n  - Barcelona (11:50 - 13:22)\n  - Mallorca (13:22 - 14:57)\n  - Granada (14:57 - 16:40)\n\n#### Thumbnails:\n- ![Thumbnail](https://i.ytimg.com/vi/L5JORXmV_A0/hqdefault.jpg)\n\n---\n\n### 2. **Spain's Coffee Scene Is CHANGING! Breakfast in Zaragoza**\n- **Channel:** Spain Speaks\n- **Views:** 5,393\n- **Duration:** 11 minutes and 15 seconds\n- **Likes:** 246\n- **Comments:** 30\n- **Chapters:**\n  - Introduction: Is Spain

# Testing the complete YouTube analysis workflow


Testing the workflow with a different YouTube URL by extracting the video ID and collecting metadata, transcript, and thumbnails. Combining the retrieved data into a comprehensive prompt, generating a summary and analysis in a single LLM invocation, and displaying the results.


In [ ]:

youtube_url = "https://www.youtube.com/watch?v=1aM1KYvl4Dw"


In [ ]:

video_id = extract_video_id.run(youtube_url)
print(f"Extracted video ID: {video_id}")

Extracted video ID: 1aM1KYvl4Dw

In [ ]:

video_metadata = get_full_metadata.run(youtube_url)
print(f"Retrieved metadata for: {video_metadata['title']}")

Retrieved metadata for: You don't understand AI until you watch this

In [ ]:

transcript = fetch_transcript.run(video_id)
print(f"Retrieved transcript with {len(transcript)} characters")

Retrieved transcript with 36107 characters

In [ ]:
thumbnails = get_thumbnails.run(youtube_url)
print(f"Retrieved {len(thumbnails)} thumbnails")

Retrieved 42 thumbnails

In [ ]:
prompt = f"""
Please analyze this YouTube video and provide a comprehensive summary.

VIDEO TITLE: {video_metadata['title']}
CHANNEL: {video_metadata['channel']}
VIEWS: {video_metadata['views']}
DURATION: {video_metadata['duration']} seconds
LIKES: {video_metadata['likes']}

TRANSCRIPT EXCERPT:
{transcript[:3000]}... (transcript truncated for brevity)

Based on this information, please provide:
1. A concise summary of the video content (3-5 bullet points)
2. The main topics or themes discussed
3. The intended audience for this content
4. A brief analysis of why this video might be performing well (or not)
"""

In [ ]:
messages = [HumanMessage(content = prompt)]
response = llm.invoke(messages)

In [ ]:

print("\n===== VIDEO ANALYSIS =====\n")
print(response.content)


===== VIDEO ANALYSIS =====

### Summary of the Video Content
1. **Fundamentals of AI**: The video explains how AI functions, focusing on neural networks and their structural similarities to the human brain.
2. **AI Learning and Content Creation**: It addresses how AI learns, specifically in applications like chatbots (e.g., ChatGPT) and image generation tools, as well as the debates surrounding AI's impact on creative fields like art and journalism.
3. **Legal and Ethical Concerns**: The video highlights controversies, including lawsuits from organizations like The New York Times claiming that AI systems are copying content and concerns from artists about losing their intellectual property.
4. **AI Capabilities**: It discusses the potential of AI in solving complex problems, such as breaking encryption and mathematical challenges, exploring the idea of AI surpassing human capabilities.
5. **Consciousness of AI**: The video speculates on whether AI could ever achieve self-awareness or 